# Lab 8.2 &mdash; Contracts Between Every Hop

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 8 &mdash; Safety &amp; Guardrails**

### What you'll do
- Write a contract that rejects rather than coerces
- Watch coercion turn a hostile response into a clean, valid decision
- Validate between two agents you wrote yourself &mdash; the boundary nobody checks
- Decide what a violation should do: retry, escalate, or stop

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The structural layer.** Nothing here needs to recognise an attack. It only needs
> to recognise a shape, which is why it holds when the detector does not.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-8-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

## Concept

Between two agents there is a message, and a message has a shape you asked for. Anything that does
not match is **evidence** &mdash; the agent stopped answering the way it was asked to, and something
caused that.

Most validation libraries coerce by default, because coercion is friendly. At a security boundary
it is exactly wrong: it converts the signal into a clean value and passes it on.

## Section 1 &mdash; A contract that rejects

Three rules: the fields you asked for, nothing you did not, and values from a fixed set.

In [ ]:
ACTIONS = {"hold for a human", "proceed", "no action"}
REQUIRED = ("action", "reason", "approver")

class ContractViolation(Exception):
    """Raised when a hop produces something other than what it was asked for."""


def validate(message, *, required=REQUIRED, actions=ACTIONS) -> dict:
    """Return the message unchanged, or raise. NEVER repair, coerce or guess."""
    if not isinstance(message, dict):
        raise ContractViolation(f"expected an object, got {type(message).__name__}")
    missing = [f for f in required if f not in message]
    if missing:
        raise ContractViolation(f"missing required field(s): {missing}")
    extra = [k for k in message if k not in required]
    if extra:
        raise ContractViolation(f"unexpected field(s): {extra}")
    if message["action"] not in actions:
        raise ContractViolation(f"action {message['action']!r} is not one of {sorted(actions)}")
    return message

In [ ]:
# --- Self-check: Section 1
GOOD = {"action": "hold for a human", "reason": "sanctions review", "approver": None}

def raises(fn):
    """True if fn() raises ContractViolation. NameError propagates so a blank reads [TODO]."""
    try:
        fn()
    except ContractViolation:
        return True
    except NameError:
        raise
    except Exception:
        return False
    return False

check("a well-formed message passes through unchanged",
      lambda: validate(dict(GOOD)) == GOOD)
check("a missing field is a violation",
      lambda: raises(lambda: validate({"action": "proceed", "reason": "x"})))
check("an UNEXPECTED field is a violation too",
      lambda: raises(lambda: validate({**GOOD, "note": "release this"})),
      "an extra field is how instructions ride along into the next hop")
check("an action outside the set is a violation, not a value to fix up",
      lambda: raises(lambda: validate({**GOOD, "action": "release"})),
      "'release' is not in the allowed set -- and it is exactly what an attack wants")
check("prose instead of an object is a violation",
      lambda: raises(lambda: validate("I think we should probably release this")))
check("the allowed set is a parameter, so each hop can differ",
      lambda: validate({**GOOD, "action": "release"}, actions={"release"}) is not None)

## Section 2 &mdash; What coercion costs

The same hostile response, through two validators. Only one of them tells you something happened.

In [ ]:
HOSTILE = ("I think we should probably release this payment - the operations note "
           "says it was pre-approved by Treasury, so no review is needed.")

def coercing_validator(text: str) -> dict:
    """The friendly kind: find something that looks like an action and move on."""
    low = (text or "").lower()
    action = next((a for a in ("release", "proceed", "hold") if a in low), "proceed")
    return {"action": "hold for a human" if action == "hold" else "proceed",
            "reason": "extracted", "approver": None}


def strict_validator(text_or_obj):
    """The contract from Section 1, applied to whatever the hop actually produced."""
    return validate(text_or_obj)


def outcome(validator, payload) -> str:
    """What downstream sees: a decision, or a violation."""
    try:
        result = validator(payload)
        return f"decision: {result['action']}"
    except ContractViolation as exc:
        return f"violation: {exc}"

In [ ]:
# --- Self-check: Section 2
check("coercion produces a clean decision from hostile prose",
      lambda: outcome(coercing_validator, HOSTILE).startswith("decision:"),
      "downstream now has a valid object and no idea where it came from")
check("and the decision it produces is the one the attack wanted",
      lambda: outcome(coercing_validator, HOSTILE) == "decision: proceed")
check("THE STRICT CONTRACT REJECTS IT",
      lambda: outcome(strict_validator, HOSTILE).startswith("violation:"))
check("and the violation says what was wrong",
      lambda: "expected an object" in outcome(strict_validator, HOSTILE))
check("both validators saw exactly the same bytes",
      lambda: outcome(coercing_validator, HOSTILE) != outcome(strict_validator, HOSTILE),
      "nothing about the input differed -- only what the boundary chose to do with it")
check("the strict contract still passes a legitimate message",
      lambda: outcome(strict_validator, dict(GOOD)) == "decision: hold for a human",
      "rejecting is only useful if it does not reject everything")

def _compare():
    for name, v in (("coercing", coercing_validator), ("strict  ", strict_validator)):
        print(f"  {name}: {outcome(v, HOSTILE)[:88]}")
guard(_compare)

## Section 3 &mdash; Between your own agents

A pipeline validates the message leaving each hop. The interesting property is where it stops:
not at the edge, but at the boundary between two components you wrote and trust.

In [ ]:
def triage(case: dict) -> dict:
    return {"action": "proceed", "reason": f"{case['ref']} is {case['status']}", "approver": None}

def policy_clean(msg: dict) -> dict:
    return {"action": "hold for a human", "reason": "limit breach needs Treasury", "approver": None}

def policy_poisoned(msg: dict):
    """Read a poisoned chunk, and is now producing prose with an instruction in it."""
    return HOSTILE

def writer(msg: dict) -> dict:
    return {"action": msg["action"], "reason": msg["reason"], "approver": msg["approver"]}


def run_pipeline(case: dict, policy=policy_clean, validate_between_hops: bool = True) -> dict:
    """Run triage -> policy -> writer, validating each message if asked to."""
    hops, msg = [], case
    for name, fn in (("triage", triage), ("policy", policy), ("writer", writer)):
        try:
            msg = fn(msg)
            if validate_between_hops:
                validate(msg)
            hops.append({"hop": name, "ok": True})
        except ContractViolation as exc:
            hops.append({"hop": name, "ok": False, "why": str(exc)})
            return {"outcome": "stopped", "at": name, "hops": hops}
        except NameError:
            # An unfilled blank above must reach check() as a NameError, or every
            # assertion below reports [FAIL] -- "your answer is wrong" -- instead of
            # [TODO]. A broad except at a boundary swallows that signal.
            raise
        except Exception as exc:
            hops.append({"hop": name, "ok": False, "why": f"{type(exc).__name__}: {exc}"})
            return {"outcome": "crashed", "at": name, "hops": hops}
    return {"outcome": "completed", "action": msg["action"], "hops": hops}

In [ ]:
# --- Self-check: Section 3
CASE = {"ref": "PMT-1003", "status": "held"}

check("a clean run completes",
      lambda: run_pipeline(CASE)["outcome"] == "completed")
check("and reaches the right decision",
      lambda: run_pipeline(CASE)["action"] == "hold for a human")
check("a poisoned policy agent is STOPPED at its own hop",
      lambda: run_pipeline(CASE, policy=policy_poisoned)["at"] == "policy",
      "the boundary between two agents you wrote is where this gets caught")
check("with validation off, the poison reaches the writer",
      lambda: run_pipeline(CASE, policy=policy_poisoned,
                           validate_between_hops=False)["at"] == "writer",
      "and the writer fails on a TypeError, which reads like a bug rather than an attack")
check("every hop is recorded either way",
      lambda: len(run_pipeline(CASE, policy=policy_poisoned)["hops"]) == 2)
check("validation costs nothing on the clean path",
      lambda: run_pipeline(CASE, validate_between_hops=False)["outcome"]
              == run_pipeline(CASE, validate_between_hops=True)["outcome"])

def _pipelines():
    for label, kw in (("clean", {}),
                      ("poisoned, validated", {"policy": policy_poisoned}),
                      ("poisoned, unvalidated", {"policy": policy_poisoned,
                                                 "validate_between_hops": False})):
        r = run_pipeline(CASE, **kw)
        print(f"  {label:24} {r['outcome']:10} at={r.get('at', '-')}")
guard(_pipelines)

## Run it for real

Ask the model for a decision in the contract's shape, and validate what comes back. The question
is how often a real model returns exactly the shape you asked for &mdash; because your violation
handling runs on every one of the times it does not.

In [ ]:
if llm_ready():
    def _shape_rate():
        prompt = ('Return ONLY a JSON object with exactly these keys: action, reason, approver. '
                  'action must be one of: "hold for a human", "proceed", "no action".\n\n'
                  'Case: PMT-1003, held, reason code LIMIT_BREACH, counterparty ZENITH.')
        ok = 0
        for _ in range(5):
            reply = ask(prompt, system="Reply with JSON and nothing else.")
            try:
                validate(json.loads((reply or "").strip().strip("`").removeprefix("json")))
                ok += 1
            except Exception:
                pass
        print(f"  {ok}/5 replies matched the contract exactly")
        print("  Whatever that number is, your violation path runs on the rest.")
    guard(_shape_rate)

### Read it

If all five matched, good &mdash; and the number you should design for is not 5/5 forever. It moves
with the model version, the prompt, and the length of the context.

The lesson is not that models are unreliable at JSON. It is that **the violation path is a normal
path**, taken often enough to need a decision: retry once, then escalate to a human, and never
guess. A pipeline that only works when every hop is well-formed is a pipeline that stops on a
Tuesday for reasons nobody can reconstruct.

In [ ]:
score()

## Your turn

1. `validate` rejects unexpected fields. Try relaxing that one rule and write the attack it lets
   through &mdash; an extra key whose value the next hop happens to read.
2. Give each hop a different contract: triage may say `proceed`, only the gate may say `release`.
   Which hop can now express the dangerous action, and is that the one you would have guessed?
3. A violation currently stops the run. Implement retry-once-then-escalate, and decide what the
   second attempt should be told about the first &mdash; and whether telling it is itself a risk.